In [1]:
import chromadb
from chromadb.utils import embedding_functions

In [2]:
# Define embedding function
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

c:\Users\mbial\AppData\Local\pypoetry\Cache\virtualenvs\gen-ai-common-use-HcUDhRlE-py3.11\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\mbial\AppData\Local\pypoetry\Cache\virtualenvs\gen-ai-common-use-HcUDhRlE-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mbial\AppData\Local\pypoetry\Cache\virtualenvs\gen-ai-common-use-HcUDhRlE-py3.11\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\User

In [3]:
client = chromadb.Client()

In [4]:
collection = client.create_collection(
    name="test_collection",
    metadata={"topic": "query testing"},
    configuration={
        "hnsw": {
            "space": "cosine",    # Distance metric
        },
        "embedding_function": ef
    }
)

In [5]:
collection.add(
    documents=[
        "Document text 1",
        "Document text 2"
    ],
    metadatas=[
        {"source": "source1", "category": "type1"},
        {"source": "source2", "category": "type2"}
    ],
    ids=["id1", "id2"]
)

In [7]:
# Get all documents
all_items = collection.get()
# Get with metadata filter
filtered_items = collection.get(
    where={"source": "source2"}
)
print("All items:", all_items)
print("Filtered items:", filtered_items)

All items: {'ids': ['id1', 'id2'], 'embeddings': None, 'documents': ['Document text 1', 'Document text 2'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'type1', 'source': 'source1'}, {'source': 'source2', 'category': 'type2'}]}
Filtered items: {'ids': ['id2'], 'embeddings': None, 'documents': ['Document text 2'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'type2', 'source': 'source2'}]}


In [ ]:
# Comparison operators
"$eq"   # equal to (string, int, float)
"$ne"   # not equal to
"$gt"   # greater than (int, float)
"$gte"  # greater than or equal to
"$lt"   # less than (int, float)
"$lte"  # less than or equal to
"$in"   # in list
"$nin"  # not in list
collection.get(
    where={
        "$and": [
            {"source": {"$eq": "langchain.com"}}, 
            {"version": {"$lt": 0.3}}
        ]
    }
)

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [9]:
# Using a list to perform an OR operation on the values of a metadata key
collection.get(
    where={
        "$and": [
            {"source": {"$in": ["langchain.com", "llamaindex.ai"]}}, 
            {"version": {"$lt": 0.3}}
        ]
    }
)

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [10]:
results = collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where={'topic': 'animals'}
)

In [11]:
results

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

In [12]:
# With document filter
results = collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where_document={'$not_contains': 'library'}
)
print(results)
# Combined filters
results = collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where={'topic': 'animals'},
    where_document={'$not_contains': 'library'}
)
print(results)

{'ids': [['id1']], 'embeddings': None, 'documents': [['Document text 1']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'source1', 'category': 'type1'}]], 'distances': [[0.9789041876792908]]}
{'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}
